# TB Portals - Kantipudi **A1** baseline (lung-seg + YOLOv5 lesion det + cavity)

Detection-based ALP = |lesion boxes ∩ MedSAM lung| / |lung|, + cavity classifier -> Timika. **This is the paper's WORST approach and the heaviest to run.** Also attach **tbx-11** (TBX11K) for the lesion detector. **Attach these Kaggle datasets before running:** `tb-portals-cxr-pngs`, `medsam-vit-b`.

## 0 - Clone the codebase

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + '/scripts'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)

## Install deps

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg", "ultralytics"], check=False)
print("deps installed")

## Paths

Edit dataset slugs if yours differ.

In [ ]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
OUT_DIR        = f"{WORK}/checkpoints/paper_a1"
os.makedirs(OUT_DIR, exist_ok=True)
TBX_ROOT  = "/kaggle/input/datasets/usmanshams/tbx-11/TBX11K"
YOLO_DIR  = f"{WORK}/tbx11k_yolo"
YOLO_BEST = f"{WORK}/yolo_runs/tbx11k/weights/best.pt"
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [ ]:
import os, sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from cache_lung_crops import main as crops_main
argv = ['--manifest', PAPER_MANIFEST, '--out-dir', CROPS_DIR,
        '--medsam-ckpt', MEDSAM_CKPT, '--size', '224', '--pad', '32']
if os.path.isfile(LUNG_DECODER):
    argv += ['--lung-decoder-ckpt', LUNG_DECODER]
crops_main(argv)
print('crops ->', CROPS_DIR, '| count:', len(os.listdir(CROPS_DIR)))

## 3 - Inspect TBX11K layout

The converter auto-discovers VOC XML. If this print shows a different structure (e.g. COCO JSON), tell me and I'll adjust the converter.

In [ ]:
import os
print("TBX_ROOT exists:", os.path.isdir(TBX_ROOT))
if os.path.isdir(TBX_ROOT):
    for name in sorted(os.listdir(TBX_ROOT))[:20]:
        sub = os.path.join(TBX_ROOT, name)
        kids = sorted(os.listdir(sub))[:6] if os.path.isdir(sub) else "(file)"
        print(" ", name, "->", kids)

## 4 - Convert TBX11K -> YOLO format (paper's official split)

Uses `lists/TBX11K_train.txt` / `TBX11K_val.txt` intersected with the annotated TB images (reproduces the paper's detection split, ~511 train / ~128 val). Watch the printed counts; if `train`/`val` come out near 0, the list format differs - paste the output to me.

In [ ]:
import sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from prepare_tbx11k_yolo import main as prep_main
prep_main(['--tbx-root', TBX_ROOT, '--out', YOLO_DIR,
           '--train-list', f"{TBX_ROOT}/lists/TBX11K_train.txt",
           '--val-list',   f"{TBX_ROOT}/lists/TBX11K_val.txt"])

## 5 - Train YOLOv5 lesion detector on TBX11K (~30-60 min)

Paper used YOLOv5n. Epochs reduced from 1000 to 100 for Kaggle; increase if time allows.

In [ ]:
from ultralytics import YOLO
yolo = YOLO("yolov5nu.pt")
yolo.train(data=f"{YOLO_DIR}/tbx11k.yaml", epochs=100, imgsz=640, batch=16,
           project=f"{WORK}/yolo_runs", name="tbx11k", exist_ok=True)
print("best weights ->", YOLO_BEST)

## 6 - A1 eval: detection ALP + cavity -> Timika (~1.5-2 h)

In [ ]:
from src.training.train_a1_detect import main as a1_main
a1_main(['--manifest', PAPER_MANIFEST, '--crops-dir', CROPS_DIR,
         '--yolo-weights', YOLO_BEST, '--medsam-ckpt', MEDSAM_CKPT, '--lung-decoder-ckpt', LUNG_DECODER,
         '--out-dir', OUT_DIR, '--held-outs','Romania','Moldova','Kazakhstan','--seeds','0','1','2',
         '--epochs','30','--batch-size','60','--accum-steps','5','--num-workers','2',
         '--cavity-no-lung-crop'])

## 7 - Save outputs

In [ ]:
!cd /kaggle/working && zip -j results_a1.zip checkpoints/paper_a1/results_a1.csv tbportals_manifest_paper.csv
!cd /kaggle/working && zip -r -q checkpoints_a1.zip checkpoints/paper_a1
print("Saved: results_a1.zip, checkpoints_a1.zip")